In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import dendrogram, linkage

np.random.seed(42)

iris = load_iris()
X = iris.data

df = pd.DataFrame(X, columns=iris.feature_names)
print("Dataset o‘lchami:", X.shape)
display(df.head())

In [ ]:
print(df.info())
display(df.describe())
print("Missing values:")
print(df.isnull().sum())

In [ ]:
plt.figure(figsize=(7,5))
plt.scatter(X[:,2], X[:,3], s=45)
plt.xlabel("Petal length")
plt.ylabel("Petal width")
plt.title("Klasterlashdan oldingi ma’lumotlar")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
print(X_scaled[:5])

In [ ]:
inertias = []
K_values = range(1, 11)

for k in K_values:
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    model.fit(X_scaled)
    inertias.append(model.inertia_)

plt.figure(figsize=(7,5))
plt.plot(K_values, inertias, marker="o")
plt.xlabel("Klasterlar soni K")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
scores = []

for k in range(2, 7):
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    scores.append([k, score])

score_df = pd.DataFrame(scores, columns=["K", "Silhouette Score"])
display(score_df)

plt.plot(score_df["K"], score_df["Silhouette Score"], marker="o")
plt.xlabel("K")
plt.ylabel("Silhouette Score")
plt.title("Klaster sifatini baholash")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

df_result = df.copy()
df_result["Cluster"] = clusters
display(df_result.head())
print(df_result["Cluster"].value_counts().sort_index())

In [ ]:
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=clusters, s=60)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("K-means klasterlari (PCA 2D)")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
print("Centroidlar:")
print(kmeans.cluster_centers_)

cluster_summary = df_result.groupby("Cluster").mean()
display(cluster_summary)

cluster_summary.T.plot(kind="bar", figsize=(10,5))
plt.title("Klasterlar xususiyatlarini taqqoslash")
plt.ylabel("O‘rtacha qiymat")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
hier = AgglomerativeClustering(n_clusters=3)
hier_labels = hier.fit_predict(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=hier_labels, s=60)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Ierarxik klasterlash natijasi")
plt.grid(alpha=0.3)
plt.show()

In [ ]:
Z = linkage(X_scaled, method="ward")

plt.figure(figsize=(11,5))
dendrogram(Z, truncate_mode="level", p=5)
plt.xlabel("Namunalar")
plt.ylabel("Masofa")
plt.title("Ierarxik klasterlash dendrogrammasi")
plt.show()

In [ ]:
kmeans_s = silhouette_score(X_scaled, clusters)
hier_s = silhouette_score(X_scaled, hier_labels)

result = pd.DataFrame({
    "Algoritm": ["K-means", "Hierarchical"],
    "Silhouette Score": [kmeans_s, hier_s]
})

display(result)